In [ ]:
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

from nltk.tokenize import word_tokenize
import re

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer

from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

In [ ]:
all_text = pd.read_parquet('clean_text_exploded.parquet').reset_index()
all_docs = all_text['Cleaned_Text'].to_list()
all_docs = [doc if doc.endswith('. ') else doc + '. ' for doc in all_docs]

### We first convert the text to a vector embedding using the all-MiniLM-L6-v2 sentence transformer.

In [ ]:
all_embeddings, exploded_data = [],[]

sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device="mps")
batch_size = 5000
count = 0
total_time = 0

for start_idx in range(0, len(all_docs), batch_size):
    count += 1
    start = time.time()
    #####################################################
    end_idx = min(start_idx + batch_size, len(all_docs))
    batch_docs = all_docs[start_idx:end_idx]
    embeddings = sentence_model.encode(batch_docs, normalize_embeddings=False, show_progress_bar=True)
    all_embeddings.extend(embeddings)
    #####################################################
    end = time.time()
    total_time += (end - start)
    clear_output(wait=True)
    #####################################################
    print("Embedding batch: " + str(int((count))+1) + "/" + str(int(len(all_docs)/batch_size)))
    print(f"Average iteration runtime: {(total_time/count) :.0f} seconds")
    print(f"Total runtime: {total_time /60 :.2f} minutes")
    est_time_remaining = (total_time / count) * ((len(all_docs) - count - batch_size) // batch_size) / 60
    print(f"Estimated time remaining: ~{est_time_remaining:.2f} minutes OR {est_time_remaining/60:.1f} hours")
    est_finish_time = (datetime.now() + timedelta(minutes=est_time_remaining))
    print(f"Estimated completion time: {est_finish_time.strftime('%I:%M %p')}")

embeddings = np.array(all_embeddings)
np.save('alltext_embeddings.npy', embeddings)

### We now perform dimensionality reduction using PCA and then normalize the reduced embeddings using L2-normalization

In [ ]:
print("Computing corpus information for coherence measures.")
tokenized_docs = [word_tokenize(doc.lower()) for doc in all_docs]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

# Models defined once outside the loop
sentence_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device="mps")
ctfidf_model        = ClassTfidfTransformer(reduce_frequent_words=True)

# We model with unigrams and bigrams, as unigrams alone may not 
# hold as semantically meaningful detail. Since the text is at the sentence level, 
# trigrams would likely result in too many singleton tokens.
vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=5)

print("Performing PCA and transforming data.")
dim_model           = PCA(n_components=0.97, svd_solver="full", random_state=39)
reduced_embeddings  = dim_model.fit_transform(embeddings)

print("Normalizing reduced embeddings using L2 Normalization.")
reduced_embeddings  = normalize(reduced_embeddings, norm="l2", axis=1)
dim_model           = BaseDimensionalityReduction()
representation_model = {
    'keybert': KeyBERTInspired(),
    'mmr':     MaximalMarginalRelevance(diversity=0.3)
}

np.save('reduced_embeddings.npy', reduced_embeddings)

### Based on the parameter selection below, we then run coherence experiments across the parameter grid to identify the best performing model.

In [ ]:
def run_topic_modeling(documents_list):
    # Prompt for algorithm and parameters
    algo_choice = input(
        "Which clustering algorithm do you want to use?\n"
        "1) KMeans\n" 
        "Press 1 to Continue"
    )
    if algo_choice == "1":
        # KMeans parameters: range for k values and list of ngram max lengths.
        k_vals_str = input(
            "Enter a range for k values as three comma-separated numbers (start, stop, step), e.g., 10,100,10: ")
        
        start_val, stop_val, step_val = map(int, k_vals_str.split(','))
        k_vals = list(range(start_val, stop_val, step_val))
    else:
        print("Invalid choice. Please press 1 to continue.")
    
    return k_vals
    
coherence_params = run_topic_modeling(all_docs) 

num_params = len(coherence_params)
total_time = 0.0
n_clust, all_text = [], []
cv_list    = []

for idx, k in enumerate(coherence_params, start=1):
    start_time = time.time()
    n_clust.append(k)

    # set up clustering
    cluster = KMeans(n_clusters=k, random_state=39, init='k-means++', n_init=5, max_iter=100, 
                     algorithm="elkan")

    model = BERTopic(
        low_memory=True,
        embedding_model=sentence_model,      # unused when passing embeddings, but kept
        umap_model=dim_model,
        hdbscan_model=cluster,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        verbose=True
    )

    topics, _ = model.fit_transform(all_docs, embeddings=reduced_embeddings)
    topics_dict = model.get_topics()
    topic_words = [word for topic_number, topic_content in model.get_topics().items() for word, _ in topic_content]

    print("Starting c_v coherence calculations")
    cv = CoherenceModel(topics=[topic_words], texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    cv_list.append(cv.get_coherence())

    # timekeeping & feedback
    iteration_time = time.time() - start_time
    total_time += iteration_time
    avg_time = total_time / idx
    rem_time = avg_time * (num_params - idx)

    clear_output(wait=True)
    print(f"Iteration {idx}/{num_params}")
    print(f"Runtime: {iteration_time/60:.2f} min (avg {avg_time/60:.2f} min),"
          f" est. remaining {rem_time/60:.2f} min")

# Plot K and resulting C_V coherence, identify best K
opt_cv_idx = np.argmin(np.abs(np.array(cv_list) - 1))  # cv    closest to 1
opt_cv_k   = n_clust[opt_cv_idx]
print(f"Optimal k (C_V): {opt_cv_k}")
 
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(n_clust, cv_list, marker='s', linestyle='-', label='C_V Coherence')
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("C_V Coherence Score")
ax.axvline(opt_cv_k, color='blue', linestyle='--', label=f"Optimal k (C_V): {opt_cv_k}")
ax.legend(loc='best')
plt.title("C_V Coherence by Number of Clusters")
plt.show()